project_root/
│
├── agent_factory/
│  
│   ├── dynamic_agent_factory.py   ← 🧠 The reusable factory
│
├── mcp_clients/
│   ├── mcp_session.py
│   ├── universal_mcp_client.py
│---mcp_servers   
│    ----- math_server.py
│
├── configs/
│   ├── settings.py
│
├── main.py

# agent_factory/dynamic_agent_factory.py
import importlib
from typing import Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, Tool
from mcp_clients.universal_mcp_client import load_all_mcp_tools


class DynamicAgentFactory:
    """
    Dynamically creates and manages LangChain/LangGraph agents
    based on a provided configuration.

    Each agent can be linked to different MCP servers, LLM models,
    and system prompts dynamically.
    """

    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.registry = {}

    async def create_agent(self, name: str):
        """
        Create and register an agent dynamically based on config.
        """
        if name not in self.config:
            raise ValueError(f"Agent '{name}' not found in configuration.")

        agent_cfg = self.config[name]
        print(f"🔧 Creating agent: {name}")
        print(f"   ↳ LLM: {agent_cfg['llm_model']}")
        print(f"   ↳ MCP Servers: {agent_cfg['mcp_servers']}")

        # 1️⃣ Load LLM
        llm = ChatOpenAI(model=agent_cfg["llm_model"], temperature=0.2)

        # 2️⃣ Dynamically import tools from MCP clients
        tools = []
        try:
            tools = await load_all_mcp_tools(agent_cfg)
        except ModuleNotFoundError:
            print(f"MCP client not found: {name} agent")
       
        # 3️⃣ Initialize agent
        agent = initialize_agent(
            tools=tools,
            llm=llm,
            agent="zero-shot-react-description",
            verbose=True,
        )

        # 4️⃣ Store in registry
        self.registry[name] = {
            "llm": llm,
            "tools": tools,
            "agent": agent,
            "system_prompt": agent_cfg.get("system_prompt", ""),
        }

        return agent

    async def get_agent(self, name: str):
        """
        Retrieve an agent from the registry or create it if missing.
        """
        if name not in self.registry:
            await self.create_agent(name)
        return self.registry[name]["agent"]

# config/settings.py
import os
from dotenv import load_dotenv

# Load .env file from project root
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
DEFAULT_MODEL = "gpt-4o"  # change to "gpt-4.1-mini" if you want cheaper


AGENT_CONFIG = {
    "agent1": {
        "llm": "openai",
        "llm_model": "gpt-4o-mini",
        "system_prompt": "You are a math expert. Use math tools wisely.",
        "mcp_servers": ["math-mcp"]
    },
   
}        
# mcp_clients/mcp_session.py
import asyncio
import sys
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

class MCPSession:
    """Manage a single MCP session for multiple tool calls"""

    def __init__(self, server_script_path: str):
        self.server_script_path = server_script_path
        self.server_name = None
        self.session = None
        self.tools_info = []
        self._stdio_ctx = None
        self._client_session_ctx = None
        self._stdio_pair = None

    async def __aenter__(self):
        server_params = StdioServerParameters(
            command=sys.executable,
            args=[self.server_script_path]
        )

        self._stdio_ctx = stdio_client(server_params)
        self._stdio_pair = await self._stdio_ctx.__aenter__()
        read, write = self._stdio_pair

        self._client_session_ctx = ClientSession(read, write)
        self.session = await self._client_session_ctx.__aenter__()

        init_result = await self.session.initialize()
        list_result = await self.session.list_tools()
        self.tools_info = list_result.tools
        self.server_name  = getattr(getattr(init_result, "serverInfo", {}), "name", None)
        print(f"✅ Connected to MCP server [{os.path.basename(self.server_script_path)}], found {len(self.tools_info)} tools")
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        if self._client_session_ctx:
            await self._client_session_ctx.__aexit__(exc_type, exc_val, exc_tb)
            self.session = None
        if self._stdio_ctx:
            await self._stdio_ctx.__aexit__(exc_type, exc_val, exc_tb)

    async def call_tool(self, tool_name: str, **kwargs):
        if not self.session:
            raise RuntimeError("Session not initialized. Use async context manager.")

        result = await self.session.call_tool(tool_name, kwargs)
        if hasattr(result, "content") and result.content:
            return result.content[0].text
        return str(result)

    def get_tool_names(self):
        return [tool.name for tool in self.tools_info]

 # mcp_client/universal_mcp_client.py
import asyncio
import os
from langchain.tools import Tool
from mcp_clients.mcp_session import MCPSession

async def load_tools_from_mcp(server_script_path: str):
    """Dynamically connect to an MCP server and create LangChain tools for its functions."""
    tools = []
    async with MCPSession(server_script_path) as session:
        for tool_info in session.tools_info:
            tool_name = tool_info.name
            desc = tool_info.description or "No description available"
            server_name = session.server_name

            async def tool_func(**kwargs):
                return await session.call_tool(tool_name, **kwargs)

            # Create synchronous wrapper for LangChain compatibility
            # def sync_tool_func(**kwargs):
            #     return asyncio.run(tool_func(**kwargs))
            
            #working-1
            
            def sync_tool_func(*args, **kwargs):
                # Handle both (a, b) and {"a": a, "b": b} cases
                if len(args) == 1 and isinstance(args[0], dict):
                    kwargs = args[0]
                elif len(args) == 2 and not kwargs:
                # LangChain sometimes passes (5, 3)
                    kwargs = {"a": args[0], "b": args[1]}

                async def run_tool():
                    return await tool_func(**kwargs)

                try:
                    # Get the current loop if it exists
                    loop = asyncio.get_running_loop()
                    return loop.create_task(run_tool())  # schedule coroutine
                except RuntimeError:
                    # No event loop is running
                    return asyncio.run(run_tool())

            tools.append(Tool(name=tool_name, func=sync_tool_func, description=desc))

    print(f"✅ Created {len(tools)} LangChain tools from {os.path.basename(server_script_path)}")
    return tools


async def load_all_mcp_tools(agent_cfg: dict):
    """Scan a folder for MCP servers and load all tools dynamically."""
    all_tools = []
    for mcp_name in agent_cfg["mcp_servers"]:
        
        current_dir = os.path.dirname(os.path.abspath(__file__))
        server_path = os.path.join(current_dir, "..", "mcp_servers", "math_server.py")
        server_path = os.path.abspath(server_path)
    
        try:
            tools = await load_tools_from_mcp(server_path)
            all_tools.extend(tools)
        except Exception as e:
                print(f"⚠️ Could not load tools from {os.path.basename(server_path)}: {e}")
    

    print(f"🔧 Total MCP tools loaded: {len(all_tools)}")
    return all_tools   

  from fastmcp.server import FastMCP

mcp = FastMCP("math-mcp")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers together. Also responds to queries like 'sum', 'plus', 'total', or 'combine'."""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers. Also responds to queries like 'product', 'times', or 'multiply'."""
    return a * b

if __name__ == "__main__":
    print("Starting server")
    mcp.run()
# main.py
import asyncio
from agent_factory.dynamic_agent_factory import DynamicAgentFactory
from config.settings import AGENT_CONFIG

async def main():
    factory = DynamicAgentFactory(AGENT_CONFIG)

    print("\n🤖 Dynamic Agent System Started")
    print("Available agents:", ", ".join(AGENT_CONFIG.keys()))
    print("Type 'switch' to change agent or 'exit' to quit.\n")

    current_agent_name = input("Enter agent name to activate: ").strip()

    while True:
        # Exit or switch logic
        if current_agent_name.lower() in ["exit", "quit"]:
            print("👋 Exiting. Goodbye!")
            break

        try:
            # Create or get agent
            agent = await factory.get_agent(current_agent_name)
            query = input(f"[{current_agent_name}] > ").strip()

            if query.lower() in ["exit", "quit"]:
                print("👋 Exiting. Goodbye!")
                break

            elif query.lower() == "switch":
                print("Available agents:", ", ".join(AGENT_CONFIG.keys()))
                current_agent_name = input("Enter agent name to switch to: ").strip()
                continue

            # Run the query
            print("\n🤔 Thinking...\n")
            # result = agent.run(query)
            result = await agent.invoke(query)
            print(f"🧠 Agent Response:\n{result}\n")

        except ValueError as e:
            print(f"❌ {e}")
            current_agent_name = input("Enter valid agent name: ").strip()
        except Exception as e:
            print(f"⚠️ Unexpected error: {e}")
            continue


if __name__ == "__main__":
    asyncio.run(main())

user prompt : add 5 and 10      i am not getting any result

Action: add  
Action Input: (5, 10)  
Observation: <Task pending name='Task-11' coro=<load_tools_from_mcp.<locals>.sync_tool_func.<locals>.run_tool() running at /Users/gvijaykumarachary/Desktop/MyComputer/E-Drive/DataScience/Repos/datascience-projects/DataScience-Gen_AI-Agentic_AI-Projects/Projects/w-18th-oct-mcp_server/mcp_clients/universal_mcp_client.py:33>>
Thought:It seems that the addition operation is consistently failing to complete. I will try to summarize the situation.

Thought: The addition operation for 5 and 10 is not yielding a result despite multiple attempts. I will conclude the process here.
Final Answer: Unable to compute the addition of 5 and 10 due to persistent errors.

> Finished chain.
⚠️ Unexpected error: object dict can't be used in 'await' expression